# Chow-Liu Exercise

Implement the Chow-Liu pipeline on binary variables.

## Implementation checklist (required)
Implement exactly these functions:
1. `empirical_pairwise_mi(samples, eps=1e-12)`
2. `estimate_tree_distribution(samples, edges, root=0, alpha=1.0)`
3. `kl_true_to_estimate(true_params, est_params, eps=1e-15)`

All other functions are provided as reference scaffolding.

## Learning goals
1. Sample from a known tree-structured distribution.
2. Estimate pairwise MI from data.
3. Recover a maximum spanning tree using Kruskal.
4. Fit tree parameters from counts.
5. Evaluate with tree recovery, MI error, and KL divergence.
6. Visualize behavior as sample size `l` changes.


In [ ]:
import itertools
import math
from collections import deque

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=4, suppress=True)


## Tree and True Distribution Utilities


In [ ]:
def build_adj(n, edges):
    """Build an undirected adjacency list.

    Parameters
    ----------
    n : int
        Number of nodes.
    edges : list[tuple[int, int]]
        Undirected edges.

    Returns
    -------
    list[list[int]]
        Adjacency list where `adj[i]` stores neighbors of node `i`.
    """
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj


def orient_tree(n, edges, root=0):
    """Orient an undirected tree away from a chosen root.

    Parameters
    ----------
    n : int
        Number of nodes.
    edges : list[tuple[int, int]]
        Undirected tree edges.
    root : int, default=0
        Root node.

    Returns
    -------
    tuple[list[int], list[list[int]], list[int]]
        `parent`, `children`, `order` where:
        - `parent[i]` is parent of node `i` (root has parent -1),
        - `children[i]` lists directed children of node `i`,
        - `order` is BFS order from the root.
    """
    adj = build_adj(n, edges)
    parent = [-1] * n
    children = [[] for _ in range(n)]
    order = []

    q = deque([root])
    parent[root] = -2  # temporary mark to avoid revisiting root
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            if parent[v] == -1:
                parent[v] = u
                children[u].append(v)
                q.append(v)

    parent[root] = -1
    return parent, children, order


def sample_uniform_tree(n, rng):
    """Sample a uniform labeled tree using a random Prufer sequence.

    Parameters
    ----------
    n : int
        Number of nodes.
    rng : np.random.Generator
        Random number generator.

    Returns
    -------
    list[tuple[int, int]]
        List of `n-1` undirected edges.
    """
    if n < 2:
        raise ValueError("n must be at least 2")

    prufer = rng.integers(0, n, size=n - 2)
    degree = np.ones(n, dtype=int)
    for v in prufer:
        degree[v] += 1

    leaves = sorted(np.where(degree == 1)[0].tolist())
    edges = []

    for v in prufer:
        leaf = leaves.pop(0)
        edges.append((leaf, int(v)))

        degree[leaf] -= 1
        degree[v] -= 1
        if degree[v] == 1:
            leaves.append(int(v))
            leaves.sort()

    u, w = np.where(degree == 1)[0].tolist()
    edges.append((int(u), int(w)))
    return edges


def random_tree_parameters(
    n,
    edges,
    rng,
    root=0,
    root_pmin=0.25,
    root_pmax=0.75,
    min_delta=0.25,
    max_delta=0.75,
):
    """Sample binary tree-distribution parameters with controlled dependence.

    Parameters
    ----------
    n : int
        Number of nodes.
    edges : list[tuple[int, int]]
        Undirected tree edges.
    rng : np.random.Generator
        Random number generator.
    root : int, default=0
        Root node used for directed parameterization.
    root_pmin, root_pmax : float
        Range for `P(X_root=1)`.
    min_delta, max_delta : float
        Range for dependence strength `|P(child=1|parent=1)-P(child=1|parent=0)|`.

    Returns
    -------
    dict
        Tree parameter dictionary used by `tree_prob` and sampling utilities.
    """
    parent, children, order = orient_tree(n, edges, root=root)

    root_prob = float(rng.uniform(root_pmin, root_pmax))
    cond = {}
    for node in order:
        if node == root:
            continue
        par = parent[node]

        # Draw a conditional table with a guaranteed minimum dependence.
        for _ in range(100):
            center = float(rng.uniform(0.2, 0.8))
            delta = float(rng.uniform(min_delta, max_delta))
            direction = float(rng.choice([-1.0, 1.0]))

            p0 = np.clip(center - 0.5 * direction * delta, 0.05, 0.95)
            p1 = np.clip(center + 0.5 * direction * delta, 0.05, 0.95)
            if abs(p1 - p0) >= 0.95 * min_delta:
                break
        else:
            p0, p1 = 0.25, 0.75

        table = np.zeros((2, 2), dtype=float)
        table[0, 1] = p0
        table[0, 0] = 1.0 - p0
        table[1, 1] = p1
        table[1, 0] = 1.0 - p1
        cond[(par, node)] = table

    return {
        "n": n,
        "edges": list(edges),
        "root": root,
        "parent": parent,
        "children": children,
        "order": order,
        "root_prob": root_prob,
        "cond": cond,
    }


def tree_prob(x, params):
    """Evaluate the probability of one binary assignment under tree parameters.

    Parameters
    ----------
    x : array-like of shape (n,)
        Binary state vector.
    params : dict
        Tree parameter dictionary.

    Returns
    -------
    float
        Probability mass `P(X=x)`.
    """
    x = np.asarray(x, dtype=int)
    root = params["root"]
    p = params["root_prob"] if x[root] == 1 else 1.0 - params["root_prob"]

    for (u, v), table in params["cond"].items():
        p *= table[x[u], x[v]]
    return float(p)


def sample_tree_distribution(params, num_samples, rng):
    """Sample i.i.d. draws from a binary tree-structured distribution.

    Parameters
    ----------
    params : dict
        Tree parameter dictionary.
    num_samples : int
        Number of samples.
    rng : np.random.Generator
        Random number generator.

    Returns
    -------
    np.ndarray of shape (num_samples, n)
        Binary samples.
    """
    n = params["n"]
    root = params["root"]
    parent = params["parent"]
    order = params["order"]

    x = np.zeros((num_samples, n), dtype=int)
    x[:, root] = (rng.random(num_samples) < params["root_prob"]).astype(int)

    for node in order:
        if node == root:
            continue
        p = parent[node]
        table = params["cond"][(p, node)]
        p1 = table[x[:, p], 1]
        x[:, node] = (rng.random(num_samples) < p1).astype(int)

    return x


def enumerate_binary_states(n):
    """Enumerate all binary states of length `n`.

    Parameters
    ----------
    n : int
        Number of binary variables.

    Returns
    -------
    np.ndarray of shape (2**n, n)
        All binary assignments in lexicographic order.
    """
    return np.array(list(itertools.product([0, 1], repeat=n)), dtype=int)


def true_pairwise_mi(params):
    """Compute exact pairwise MI on a tree without enumerating all 2^n states.

    Parameters
    ----------
    params : dict
        Tree parameter dictionary.

    Returns
    -------
    np.ndarray of shape (n, n)
        Symmetric mutual-information matrix with zeros on the diagonal.
    """
    n = params["n"]
    edges = params["edges"]
    root = params["root"]
    parent = params["parent"]
    order = params["order"]
    cond = params["cond"]

    # 1) Exact node marginals via top-down propagation.
    node_marg = np.zeros((n, 2), dtype=float)
    node_marg[root, 1] = params["root_prob"]
    node_marg[root, 0] = 1.0 - params["root_prob"]

    for node in order:
        if node == root:
            continue
        p = parent[node]
        table = cond[(p, node)]
        node_marg[node] = node_marg[p] @ table

    # 2) Conditional transitions for both directions of every edge.
    transition = [dict() for _ in range(n)]
    for (p, c), table in cond.items():
        transition[p][c] = table

        joint = np.zeros((2, 2), dtype=float)
        for xp in (0, 1):
            for xc in (0, 1):
                joint[xp, xc] = node_marg[p, xp] * table[xp, xc]

        rev = np.zeros((2, 2), dtype=float)
        for xc in (0, 1):
            denom = node_marg[c, xc]
            if denom > 0:
                rev[xc, 0] = joint[0, xc] / denom
                rev[xc, 1] = joint[1, xc] / denom
        transition[c][p] = rev

    adj = build_adj(n, edges)

    def path_nodes(src, dst):
        """Return the unique tree path from `src` to `dst`."""
        prev = [-1] * n
        q = deque([src])
        prev[src] = src

        while q:
            u = q.popleft()
            if u == dst:
                break
            for v in adj[u]:
                if prev[v] == -1:
                    prev[v] = u
                    q.append(v)

        path = [dst]
        while path[-1] != src:
            path.append(prev[path[-1]])
        path.reverse()
        return path

    # 3) Propagate conditionals along paths and compute MI entries.
    mi = np.zeros((n, n), dtype=float)
    for i in range(n):
        pi = node_marg[i]
        for j in range(i + 1, n):
            pj = node_marg[j]
            path = path_nodes(i, j)

            trans = np.eye(2, dtype=float)
            for u, v in zip(path[:-1], path[1:]):
                trans = trans @ transition[u][v]

            pij = np.zeros((2, 2), dtype=float)
            for a in (0, 1):
                pij[a, :] = pi[a] * trans[a, :]

            val = 0.0
            for a in (0, 1):
                for b in (0, 1):
                    if pij[a, b] > 0:
                        val += pij[a, b] * math.log(pij[a, b] / (pi[a] * pj[b]))
            mi[i, j] = mi[j, i] = val

    return mi


## Estimation: MI, Kruskal, Parameters, KL


In [ ]:
def empirical_pairwise_mi(samples, eps=1e-12):
    """Estimate pairwise mutual information from samples.

    Parameters
    ----------
    samples : np.ndarray of shape (m, n)
        Binary data matrix with `m` samples and `n` variables.
    eps : float, default=1e-12
        Small constant for numerical stability in log-ratio terms.

    Returns
    -------
    np.ndarray of shape (n, n)
        Symmetric MI estimate matrix with zeros on the diagonal.

    Notes
    -----
    Implement using empirical marginals/joints:
    - `p_i(a)`, `p_j(b)`, `p_ij(a,b)`
    - `MI(i,j) = sum_{a,b} p_ij(a,b) log(p_ij(a,b)/(p_i(a)p_j(b)))`
    """
    raise NotImplementedError("Implement empirical_pairwise_mi")


def kruskal_max_spanning_tree(weights):
    """Compute a maximum spanning tree using Kruskal's algorithm.

    Parameters
    ----------
    weights : np.ndarray of shape (n, n)
        Symmetric edge-weight matrix (here: MI estimates).

    Returns
    -------
    list[tuple[int, int]]
        `n-1` edges of a maximum spanning tree.
    """
    n = weights.shape[0]
    edges = [(weights[i, j], i, j) for i in range(n) for j in range(i + 1, n)]
    edges.sort(key=lambda t: t[0], reverse=True)

    parent = list(range(n))
    rank = [0] * n

    def find(x):
        """Find set representative with path compression."""
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        """Union-by-rank; return True iff a merge happened."""
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1
        return True

    tree = []
    for _, u, v in edges:
        if union(u, v):
            tree.append((u, v))
            if len(tree) == n - 1:
                break
    return tree


def estimate_tree_distribution(samples, edges, root=0, alpha=1.0):
    """Estimate a tree-structured distribution from counts on a fixed tree.

    Parameters
    ----------
    samples : np.ndarray of shape (m, n)
        Binary dataset.
    edges : list[tuple[int, int]]
        Undirected edges of the recovered tree.
    root : int, default=0
        Root used to orient the tree.
    alpha : float, default=1.0
        Laplace smoothing constant.

    Returns
    -------
    dict
        Parameter dictionary with the following key/value types:
        - `n` : int
        - `edges` : list[tuple[int, int]]
        - `root` : int
        - `parent` : list[int] of length `n`
        - `children` : list[list[int]] of length `n`
        - `order` : list[int] of length `n`
          Root-first traversal order used for directed-tree computations.
          It must satisfy: `order[0] == root`, and every non-root node
          appears after its parent.
        - `root_prob` : float, estimate of `P(X_root=1)`
        - `cond` : dict[tuple[int, int], np.ndarray]
          where each array has shape `(2, 2)` and dtype `float`, with
          `cond[(p, c)][xp, xc] = P(X_c=xc | X_p=xp)`.

    Notes
    -----
    Implement root and conditional estimates with Laplace smoothing.
    Keep the returned dictionary layout identical to `random_tree_parameters`.
    """
    raise NotImplementedError("Implement estimate_tree_distribution")


def kl_true_to_estimate(true_params, est_params, eps=1e-15):
    """Compute KL divergence KL(P_true || P_est).

    Parameters
    ----------
    true_params : dict
        Parameters of the data-generating tree distribution.
    est_params : dict
        Parameters of the estimated tree distribution.
    eps : float, default=1e-15
        Stability constant to avoid log of zero.

    Returns
    -------
    float
        KL divergence value `sum_x p(x) log(p(x)/q(x))`.

    """
    raise NotImplementedError("Implement kl_true_to_estimate")


def same_tree(edges_a, edges_b):
    """Check edge-set equality of two undirected trees."""
    sa = {tuple(sorted(e)) for e in edges_a}
    sb = {tuple(sorted(e)) for e in edges_b}
    return sa == sb


## Experiment Runner


In [ ]:
def run_trial(n, l, rng, alpha=0.0, diagnose_failure=True):
    """Run one full Chow-Liu trial and return evaluation metrics.

    Parameters
    ----------
    n : int
        Number of variables (tree nodes).
    l : int
        Number of sampled observations.
    rng : np.random.Generator
        Random generator used for both model and data sampling.
    alpha : float, default=0.0
        Laplace smoothing value used by the estimator.
    diagnose_failure : bool, default=True
        If True, print edge-level diagnostics when tree recovery fails.

    Returns
    -------
    dict
        Dictionary with keys: `tree_ok`, `mi_error`, `kl`.
    """
    # Ground-truth model and synthetic data.
    true_edges = sample_uniform_tree(n, rng)
    true_params = random_tree_parameters(n, true_edges, rng)
    samples = sample_tree_distribution(true_params, l, rng)

    # Chow-Liu pipeline: MI estimate -> MaxST -> parameter fitting.
    mi_hat = empirical_pairwise_mi(samples)
    est_edges = kruskal_max_spanning_tree(mi_hat)
    est_params = estimate_tree_distribution(samples, est_edges, root=0, alpha=alpha)

    # Evaluation metrics.
    mi_true = true_pairwise_mi(true_params)
    mask = ~np.eye(n, dtype=bool)
    mi_error = float(np.mean(np.abs(mi_hat - mi_true)[mask]))
    kl = kl_true_to_estimate(true_params, est_params)

    tree_ok = same_tree(true_edges, est_edges)
    if (not tree_ok) and diagnose_failure:
        true_set = {tuple(sorted(e)) for e in true_edges}
        est_set = {tuple(sorted(e)) for e in est_edges}
        missing = sorted(true_set - est_set)
        extra = sorted(est_set - true_set)

        print("Tree recovery failed.")
        print(f"  n={n}, l={l}, alpha={alpha}")
        print(f"  missing true edges ({len(missing)}): {missing}")
        print(f"  extra estimated edges ({len(extra)}): {extra}")

        if missing:
            print("  Missing-edge MI (true, estimated):")
            for u, v in missing:
                print(f"    ({u}, {v}): ({mi_true[u, v]:.4f}, {mi_hat[u, v]:.4f})")

        if extra:
            print("  Extra-edge MI (true, estimated):")
            for u, v in extra:
                print(f"    ({u}, {v}): ({mi_true[u, v]:.4f}, {mi_hat[u, v]:.4f})")

    return {
        "tree_ok": float(tree_ok),
        "mi_error": mi_error,
        "kl": kl,
    }


def run_experiment(n=8, sample_sizes=(50, 100, 200, 500, 1000, 2000, 5000), trials=10, seed=0, alpha=1.0, diagnose_failure=False):
    """Run repeated trials across sample sizes.

    Returns
    -------
    dict
        Nested metrics dictionary indexed by sample size and metric name.
    """
    rng = np.random.default_rng(seed)
    results = {L: {"tree_ok": [], "mi_error": [], "kl": []} for L in sample_sizes}

    for L in sample_sizes:
        for _ in range(trials):
            out = run_trial(n=n, l=L, rng=rng, alpha=alpha, diagnose_failure=diagnose_failure)
            for k, v in out.items():
                results[L][k].append(v)

    return results


def summarize_results(results):
    """Aggregate means and standard deviations for each metric."""
    sample_sizes = sorted(results)
    summary = {"sample_sizes": sample_sizes}
    for metric in ("tree_ok", "mi_error", "kl"):
        values = np.array([results[L][metric] for L in sample_sizes], dtype=float)
        summary[f"{metric}_mean"] = values.mean(axis=1)
        summary[f"{metric}_std"] = values.std(axis=1)
    return summary


def plot_summary(summary):
    """Plot recovery/estimation metrics as a function of sample size."""
    x = np.array(summary["sample_sizes"], dtype=float)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    specs = [
        ("tree_ok", "Tree recovery rate", "Probability"),
        ("mi_error", "MI estimation error (MAE)", "Error"),
        ("kl", "KL(P_true || P_est)", "KL divergence"),
    ]

    for ax, (name, title, ylabel) in zip(axes, specs):
        mean = summary[f"{name}_mean"]
        std = summary[f"{name}_std"]
        ax.plot(x, mean, marker="o")
        ax.fill_between(x, mean - std, mean + std, alpha=0.1)
        ax.set_xscale("log")
        ax.set_title(title)
        ax.set_xlabel("number of samples l")
        ax.set_ylabel(ylabel)

    plt.tight_layout()
    plt.show()


## Run and Plot

After filling all TODOs, run the following cell.


In [ ]:
results = run_experiment(n=10, sample_sizes=(50, 500, 5000), trials=100, seed=7, alpha=0.5)
summary = summarize_results(results)
plot_summary(summary)

for L in summary["sample_sizes"]:
    tree_rate = np.mean(results[L]["tree_ok"])
    mi_mae = np.mean(results[L]["mi_error"])
    kl_mean = np.mean(results[L]["kl"])
    print(f"l={L:4d} | tree_ok={tree_rate:6.3f} | mi_mae={mi_mae:8.5f} | kl={kl_mean:8.5f}")


In [ ]:
# Example: visualize one run_trial instance
n = 8
l = 5_000
seed = 5

# run_trial returns only metrics, so we replay the same random seed to inspect internals.
trial_metrics = run_trial(n=n, l=l, rng=np.random.default_rng(seed))
rng = np.random.default_rng(seed)
true_edges = sample_uniform_tree(n, rng)
true_params = random_tree_parameters(n, true_edges, rng)
samples = sample_tree_distribution(true_params, l, rng)
mi_hat = empirical_pairwise_mi(samples)
est_edges = kruskal_max_spanning_tree(mi_hat)
est_params = estimate_tree_distribution(samples, est_edges, root=0, alpha=0.0)
mi_true = true_pairwise_mi(true_params)

def draw_tree(ax, n, edges, title):
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    pts = np.c_[np.cos(theta), np.sin(theta)]
    for u, v in edges:
        ax.plot([pts[u, 0], pts[v, 0]], [pts[u, 1], pts[v, 1]], lw=2, color="tab:blue")
    ax.scatter(pts[:, 0], pts[:, 1], s=180, color="white", edgecolor="black", zorder=3)
    for i, (x, y) in enumerate(pts):
        ax.text(x, y, str(i), ha="center", va="center", fontsize=10, zorder=4)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
draw_tree(axes[0, 0], n, true_edges, "True tree")
draw_tree(axes[0, 1], n, est_edges, "Recovered Chow-Liu tree")

im0 = axes[1, 0].imshow(mi_true, cmap="viridis")
axes[1, 0].set_title("True pairwise MI")
axes[1, 0].set_xlabel("j")
axes[1, 0].set_ylabel("i")
plt.colorbar(im0, ax=axes[1, 0], fraction=0.046, pad=0.04)

im1 = axes[1, 1].imshow(mi_hat, cmap="viridis")
axes[1, 1].set_title("Estimated pairwise MI")
axes[1, 1].set_xlabel("j")
axes[1, 1].set_ylabel("i")
plt.colorbar(im1, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.suptitle(f"One trial (n={n}, l={l}, seed={seed})", y=0.98)
plt.tight_layout()
plt.show()

print("run_trial metrics:")
for k, v in trial_metrics.items():
    print(f"  {k}: {v:.6f}")

print("\nSanity check: recomputed KL from recovered model:")
print(f"  KL(P_true || P_est) = {kl_true_to_estimate(true_params, est_params):.6f}")
